# Training for Conductivity

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import uuid
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

import os
from helper.pipeline_utils import RegressionDataset, Net, compute_loss, compute_nth_r2



def run_pipeline(
    TRAIN_PATH: str,
    BATCH_SIZE: int,
    LR: float,
    NUM_EPOCHS: int,
    WEIGHT_DECAY: float,
    NOISE_STD: float,
    SEED: int,
    hidden_dims: list,
    activation: str,
    output_weights: list,
    use_validation: bool,
    validation_split: float,
    save_model: bool = True,
):
    """
    Full training pipeline without TEST_PATH:
      1) Load & split training CSV
      2) Scale features & targets
      3) Prepare DataLoaders (with optional noise)
      4) Build model & optimizer
      5) Train, tracking weighted MSE & R² on the 2nd output
      6) Optionally save model and params
      7) Plot losses and R² curves with run settings caption
    """
    # --- [1] Setup & reproducibility ---
    tag = uuid.uuid4().hex[:8]
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- [2] Load & optionally split data ---
    df = pd.read_csv(TRAIN_PATH)
    X_all = df.iloc[:, :14].values
    Y_all = df.iloc[:, 14:].values

    if use_validation:
        X_train, X_val, Y_train, Y_val = train_test_split(
            X_all, Y_all, test_size=validation_split, random_state=SEED
        )
    else:
        X_train, Y_train = X_all, Y_all
        X_val = Y_val = None

    # --- [3] Scale features & targets on train only ---
    x_scaler = StandardScaler().fit(X_train)
    y_scaler = StandardScaler().fit(Y_train)

    X_train = x_scaler.transform(X_train)
    Y_train = y_scaler.transform(Y_train)
    if use_validation:
        X_val = x_scaler.transform(X_val)
        Y_val = y_scaler.transform(Y_val)

    # --- [4] Prepare datasets and dataloaders ---
    train_aug_ds   = RegressionDataset(X_train, Y_train, augment=True,  noise_std=NOISE_STD)
    train_clean_ds = RegressionDataset(X_train, Y_train, augment=False)
    train_loader_aug   = DataLoader(train_aug_ds,   batch_size=BATCH_SIZE, shuffle=True)
    train_loader_clean = DataLoader(train_clean_ds, batch_size=BATCH_SIZE, shuffle=False)

    val_loader = None
    if use_validation:
        val_ds     = RegressionDataset(X_val, Y_val, augment=False)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    # --- [5] Build model & optimizer ---
    model = Net(in_dim=14, hidden_dims=hidden_dims, out_dim=6, activation_str=activation)
    model.to(device)
    weights   = torch.tensor(output_weights, dtype=torch.float32).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # --- [6] Training loop ---
    train_losses, val_losses = [], []
    r2_train, r2_val = [], []
    best_val_loss = float("inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        for xb, yb in train_loader_aug:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = ((model(xb) - yb) ** 2 * weights).mean()
            loss.backward()
            optimizer.step()

        # evaluate clean train set
        tr_loss = compute_loss(model, train_loader_clean, weights, device)
        tr_r2   = compute_nth_r2(model, train_loader_clean, y_scaler, device, index=1)
        train_losses.append(tr_loss)
        r2_train.append(tr_r2)

        if use_validation:
            v_loss = compute_loss(model, val_loader, weights, device)
            v_r2   = compute_nth_r2(model, val_loader, y_scaler, device, index=1)
            val_losses.append(v_loss)
            r2_val.append(v_r2)

        # save best model
        if save_model:
            model_file = f"Conductivity_model_{tag}.pth"
            if (not use_validation) or (use_validation and v_loss < best_val_loss):
                if use_validation:
                    best_val_loss = v_loss
                torch.save(model.state_dict(), model_file)

        # log metrics
        if use_validation:
            print(f"Epoch {epoch:3d} ‖ train_loss: {tr_loss:.4f} ‖ val_loss: {v_loss:.4f} "
                  f"‖ R²_2nd: train={tr_r2:.4f}, val={v_r2:.4f}")
        else:
            print(f"Epoch {epoch:3d} ‖ train_loss: {tr_loss:.4f} ‖ R²_2nd train={tr_r2:.4f}")

    # --- [7] Save run parameters if requested ---
    if save_model:
        params = {
            "TRAIN_PATH":       TRAIN_PATH,
            "BATCH_SIZE":       BATCH_SIZE,
            "LR":               LR,
            "NUM_EPOCHS":       NUM_EPOCHS,
            "WEIGHT_DECAY":     WEIGHT_DECAY,
            "NOISE_STD":        NOISE_STD,
            "SEED":             SEED,
            "hidden_dims":      hidden_dims,
            "activation":       activation,
            "output_weights":   output_weights,
            "use_validation":   use_validation,
            "validation_split": validation_split,
            "tag":              tag,
        }
        with open(f"Conductivity_params_{tag}.txt", "w") as f:
            for k, v in params.items():
                f.write(f"{k}: {v}\n")

    # build caption for plots
    caption = (
        f"TRAIN_PATH={TRAIN_PATH}, BATCH_SIZE={BATCH_SIZE}, LR={LR}, "
        f"NUM_EPOCHS={NUM_EPOCHS}, WEIGHT_DECAY={WEIGHT_DECAY}, NOISE_STD={NOISE_STD}, "
        f"SEED={SEED}, hidden_dims={hidden_dims}, activation={activation}, "
        f"output_weights={output_weights}, use_validation={use_validation}, "
        f"validation_split={validation_split}, tag={tag}"
    )
    epochs = np.arange(1, NUM_EPOCHS + 1)

    # --- [8] Plot Clean & Validation Loss ---
    fig, ax = plt.subplots()
    ax.plot(epochs, train_losses, label="Train Loss (clean)")
    if use_validation:
        ax.plot(epochs, val_losses, label="Validation Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Weighted MSE Loss")
    ax.set_title("Clean Train & Validation Loss")
    ax.legend()
    fig.subplots_adjust(bottom=0.25)
    fig.text(0.5, 0.01, caption, ha="center", fontsize=8, wrap=True)
    plt.show()

    # --- [9] Plot R² for Second Output ---
    fig, ax = plt.subplots()
    ax.plot(epochs, r2_train, label="Train R² (2nd)")
    if use_validation:
        ax.plot(epochs, r2_val, label="Val R² (2nd)")
        ax.axhline(0.8, linestyle="--", color="gray", label="0.8 Threshold")
    else:
        ax.axhline(0.8, linestyle="--", color="gray", label="0.8 Threshold")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("R² Score")
    ax.set_title("Second Output R² Over Epochs")
    ax.legend()
    fig.subplots_adjust(bottom=0.25)
    fig.text(0.5, 0.01, caption, ha="center", fontsize=8, wrap=True)
    plt.show()
